In [1]:
AUTH_KEY = "eaa87c666bbb49c018d00441a40a2edf85846d98d7e42203b1c31a811c4b0a9b" # 발급받은 키
LIB_CODE = "111314"  # 서울도서관
# 이진아도서관: 111051
BASE_URL = "https://data4library.kr/api/itemSrch" # 장서목록검색
LOAN_BASE_URL = "https://data4library.kr/api/loanItemSrch" # 대출목록검색

In [2]:
# libCode: 도서관 코드 (111314는 서울도서관)

# startDt: 검색 시작 일자 (YYYY-MM-DD 형식)

# endDt: 검색 종료 일자 (YYYY-MM-DD 형식)

# authKey: 발급받은 인증키

# type: 검색 조건 (선택사항, ALL은 전체 데이터)

# pageNo: 페이지 번호 (기본값 1)

# pageSize: 페이지당 결과 수 (최대 값 확인 필요)

In [3]:
import requests
import pandas as pd
from xml.etree import ElementTree as ET


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\kim\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\kim\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\kim\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.sta

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\kim\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\kim\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\kim\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.sta

AttributeError: _ARRAY_API not found

In [4]:
def fetch_library_data(start_date, end_date, page_no=1, page_size=100):
    """장서/대출 데이터 조회"""
    params = {
        'authKey': AUTH_KEY,
        'libCode': LIB_CODE,
        'startDt': start_date,
        'endDt': end_date,
        'pageNo': page_no,
        'pageSize': page_size,
        'format': 'xml'  # 또는 'json'
    }
    
    response = requests.get(BASE_URL, params=params)
    return response.content

def parse_xml_to_dataframe(xml_data):
    """XML 데이터를 DataFrame으로 변환"""
    root = ET.fromstring(xml_data)
    
    books = []
    for doc in root.findall('.//doc'):
        book_info = {}
        for item in doc.findall('.//'):
            book_info[item.tag] = item.text
        books.append(book_info)
    
    return pd.DataFrame(books)

# 데이터 수집 및 저장
xml_data = fetch_library_data('2017-06-01', '2017-06-30')
df = parse_xml_to_dataframe(xml_data)

# CSV로 저장 (장서 및 대출 정보 포함)
df.to_csv('seoul_library_data_202106.csv', index=False, encoding='utf-8-sig')

# 특정 컬럼만 분리하여 저장하고 싶다면
collection_columns = ['bookname', 'authors', 'publisher', 'publication_year', 'isbn13', 'class_no']
loan_columns = ['bookname', 'loanCnt', 'ranking']

# 필요한 컬럼이 있는 경우에만 저장
if all(col in df.columns for col in collection_columns):
    df[collection_columns].to_csv('collection_data.csv', index=False, encoding='utf-8-sig')
